In [3]:
import pandas as pd
import numpy as np
import re

# --------------------------------------------
#  Load CSV
# --------------------------------------------

df = pd.read_csv("outputs/strategy_results.csv")
df.columns = df.columns.str.strip()

# Ensure flipped column is boolean
df["flipped"] = df["flipped"].astype(str).str.lower() == "true"

In [4]:
# --------------------------------------------
# Extract Numeric Perturbation Strength
# --------------------------------------------

def extract_strength(stage_name):
    if stage_name == "original":
        return 0.0

    # Extract all numbers from stage string
    nums = re.findall(r"\d+\.\d+", stage_name)
    
    if len(nums) == 0:
        return 0.0
    
    # If hybrid stage, take first number as primary strength
    return float(nums[0])

df["strength"] = df["stage"].apply(extract_strength)

In [5]:
# --------------------------------------------
#  Sort Properly
# --------------------------------------------

df = df.sort_values(["sample_id", "strategy", "strength"])

In [8]:
# --------------------------------------------
#  Minimal Flip Detection
# --------------------------------------------

summary_rows = []

for (sample_id, strategy), group in df.groupby(["sample_id", "strategy"]):

    group = group.sort_values("strength")

    # Get baseline (original image)
    baseline_rows = group[group["stage"] == "original"]

    if len(baseline_rows) == 0:
        baseline_row = group.iloc[0]  # fallback safety
    else:
        baseline_row = baseline_rows.iloc[0]

    baseline_conf = baseline_row["confidence"]

    flipped_flag = False
    min_parameter = None
    confidence_after = None
    confidence_drop = None

    for _, row in group.iterrows():
        if row["stage"] == "original":
            continue

        if row["flipped"] == True:
            flipped_flag = True
            min_parameter = row["parameter"]
            confidence_after = row["confidence"]
            confidence_drop = baseline_conf - confidence_after
            break

    summary_rows.append({
        "sample_id": sample_id,
        "strategy": strategy,
        "min_parameter": min_parameter,
        "confidence_before": baseline_conf,
        "confidence_after": confidence_after,
        "confidence_drop": confidence_drop,
        "flipped": flipped_flag
    })

summary_df = pd.DataFrame(summary_rows)



In [9]:
# --------------------------------------------
#  Save Output
# --------------------------------------------

summary_df.to_csv("min_change_summary.csv", index=False)
print("Saved: min_change_summary.csv")


Saved: min_change_summary.csv


In [10]:
# --------------------------------------------
#  Global Fragility Metrics
# --------------------------------------------

print("\n========== GLOBAL FRAGILITY METRICS ==========\n")

# Flip rate per strategy
flip_rate = summary_df.groupby("strategy")["flipped"].mean()
print("Flip Rate per Strategy:")
print(flip_rate)
print()

# Average minimal perturbation (flipped only)
avg_min_param = (
    summary_df[summary_df["flipped"]]
    .groupby("strategy")["min_parameter"]
    .mean()
)
print("Average Minimal Perturbation to Flip:")
print(avg_min_param)
print()

# Average confidence drop
avg_conf_drop = (
    summary_df[summary_df["flipped"]]
    .groupby("strategy")["confidence_drop"]
    .mean()
)
print("Average Confidence Drop:")
print(avg_conf_drop)
print()

# Percentage never flipped
never_flipped = 1 - summary_df["flipped"].mean()
print(f"Percentage Never Flipped (All Strategies): {never_flipped:.2f}")


========== GLOBAL FRAGILITY METRICS ==========

Flip Rate per Strategy:
strategy
A_blur            0.0
A_blur_freq       0.0
B_noise           0.0
B_noise_bright    0.0
original          0.0
Name: flipped, dtype: float64

Average Minimal Perturbation to Flip:
Series([], Name: min_parameter, dtype: object)

Average Confidence Drop:
Series([], Name: confidence_drop, dtype: object)

Percentage Never Flipped (All Strategies): 1.00


Conclusion from Global Fragility Metrics

 Flip Rate = 0%
→ None of the attacks caused any prediction to change.

 Average Minimal Perturbation = N/A
→ No perturbation was enough to flip any sample.

 Confidence Drop = N/A
→ The model’s confidence stayed stable under all tested attacks.

 Percentage Never Flipped = 100%
→ All samples remained correctly classified.

 Overall
→ The detector is robust to the tested blur, noise, and brightness changes, but stronger attacks may be needed to identify potential weaknesses.